# Data Augmentation Playground
Use this notebook to tweak the golfer frame augmentation pipeline. Adjust the slider below to decide how many augmented videos you want to synthesize from each raw recording before running the helper cells that follow.

In [38]:
%pip install matplotlib seaborn pandas numpy albumentations opencv-python-headless scikit-learn ipywidgets quiet

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement quiet (from versions: none)
ERROR: No matching distribution found for quiet


In [39]:
# Core libraries
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
import albumentations as A

# Notebook widgets
from IPython.display import display
import ipywidgets as widgets

In [40]:
# Interactive control for how many augmented videos to derive per source video
video_multiplier_slider = widgets.IntSlider(
    value=4,
    min=1,
    max=15,
    step=1,
    description='Videos/source',
    continuous_update=True
 )

display(widgets.VBox([
    widgets.HTML('<b>Augmented videos per raw video</b>'),
    video_multiplier_slider
]))

def get_target_video_count(source_videos: int = 1) -> int:
    """Return the number of augmented videos created from the given source count."""
    return source_videos * int(video_multiplier_slider.value)

In [41]:
class DataAugmentor:
    def __init__(self):
        """Initialize a reusable augmentation pipeline."""
        base_transforms = [
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.GaussianBlur(blur_limit=(3, 7), p=0.3),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.RandomScale(scale_limit=0.2, p=0.5),
        ]
        # ReplayCompose lets us reapply the same sampled transform to every frame in a video.
        self.transform = A.ReplayCompose(base_transforms)

    def augment_brightness(self, image, factor=1.5):
        """Adjust image brightness via PIL enhancer."""
        pil_image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        enhancer = ImageEnhance.Brightness(pil_image)
        enhanced = enhancer.enhance(factor)
        return cv2.cvtColor(np.array(enhanced), cv2.COLOR_RGB2BGR)

    def augment_contrast(self, image, factor=1.5):
        """Adjust image contrast via PIL enhancer."""
        pil_image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        enhancer = ImageEnhance.Contrast(pil_image)
        enhanced = enhancer.enhance(factor)
        return cv2.cvtColor(np.array(enhanced), cv2.COLOR_RGB2BGR)

    def augment_zoom(self, image, zoom_factor=1.2):
        """Zoom in or out by cropping around the center."""
        h, w = image.shape[:2]
        new_h, new_w = int(h / zoom_factor), int(w / zoom_factor)
        top = (h - new_h) // 2
        left = (w - new_w) // 2
        cropped = image[top:top + new_h, left:left + new_w]
        zoomed = cv2.resize(cropped, (w, h))
        return zoomed

    def augment_rotation(self, image, angle=10):
        """Rotate the image by the provided angle."""
        h, w = image.shape[:2]
        center = (w // 2, h // 2)
        matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(image, matrix, (w, h))
        return rotated

    def augment_flip(self, image, flip_code=1):
        """Flip images horizontally, vertically, or both."""
        return cv2.flip(image, flip_code)

    def augment_noise(self, image, noise_level=25):
        """Inject Gaussian noise to simulate sensor variation."""
        noise = np.random.normal(0, noise_level, image.shape).astype(np.uint8)
        noisy = cv2.add(image, noise)
        return noisy

    def augment_all_variations(self, image, output_dir='augmented'):
        """Produce every deterministic augmentation and save previews."""
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)

        augmentations = {
            'original': image,
            'brighten': self.augment_brightness(image, 1.5),
            'darken': self.augment_brightness(image, 0.6),
            'high_contrast': self.augment_contrast(image, 1.8),
            'low_contrast': self.augment_contrast(image, 0.6),
            'zoom_in': self.augment_zoom(image, 1.3),
            'zoom_out': self.augment_zoom(image, 0.8),
            'rotate_left': self.augment_rotation(image, -15),
            'rotate_right': self.augment_rotation(image, 15),
            'flip_horizontal': self.augment_flip(image, 1),
            'noise': self.augment_noise(image, 30)
        }

        fig, axes = plt.subplots(3, 4, figsize=(16, 12))
        axes = axes.ravel()

        for idx, (name, aug_image) in enumerate(augmentations.items()):
            if idx < len(axes):
                axes[idx].imshow(cv2.cvtColor(aug_image, cv2.COLOR_BGR2RGB))
                axes[idx].set_title(name.replace('_', ' ').title())
                axes[idx].axis('off')
                cv2.imwrite(str(output_path / f'{name}.jpg'), aug_image)

        plt.tight_layout()
        plt.show()

        print(f"Generated {len(augmentations)} augmented variations")
        return augmentations

    def augment_batch(self, image_list, augmentations_per_image=5):
        """Apply random albumentations transforms to an image batch."""
        augmented_batch = []

        for img in image_list:
            augmented_batch.append(img)
            for _ in range(augmentations_per_image):
                augmented = self.transform(image=img)['image']
                augmented_batch.append(augmented)

        print(f"Original batch: {len(image_list)} images")
        print(f"Augmented batch: {len(augmented_batch)} images")
        print(f"Expansion factor: {len(augmented_batch) / len(image_list):.1f}x")
        return augmented_batch

    def augment_sequence_consistently(self, frames: list[np.ndarray]) -> list[np.ndarray]:
        """Apply one sampled ReplayCompose transform uniformly across every frame."""
        if not frames:
            return []
        target_h, target_w = frames[0].shape[:2]
        replay = self.transform(image=frames[0])
        first_frame = replay['image']
        if first_frame.shape[:2] != (target_h, target_w):
            first_frame = cv2.resize(first_frame, (target_w, target_h))
        augmented_frames = [first_frame]
        for frame in frames[1:]:
            result = A.ReplayCompose.replay(replay['replay'], image=frame)
            augmented = result['image']
            if augmented.shape[:2] != (target_h, target_w):
                augmented = cv2.resize(augmented, (target_w, target_h))
            augmented_frames.append(augmented)
        return augmented_frames

In [42]:
# Instantiate a reusable augmentor instance for downstream cells
augmentor = DataAugmentor()
print('Augmentor ready - rerun this cell if you tweak the class above.')

Augmentor ready - rerun this cell if you tweak the class above.


C:\Users\tinal\AppData\Local\Temp\ipykernel_19148\2351886767.py:7: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),


In [43]:
# Video helpers
def extract_frames_from_video(video_path: Path, sample_every: int = 1, max_frames: int | None = None):
    """Load frames from a .mov or .mp4 clip using OpenCV."""
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f'Could not open video: {video_path}')
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frames: list[np.ndarray] = []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % max(sample_every, 1) == 0:
            frames.append(frame)
            if max_frames and len(frames) >= max_frames:
                break
        frame_idx += 1
    cap.release()
    return frames, fps


def export_augmented_videos(
    frames: list[np.ndarray],
    augmentor: DataAugmentor,
    video_count: int,
    output_dir: str | Path = 'augmented_videos',
    fps: int = 30,
    base_name: str | None = None,
):
    """Write multiple augmented videos derived from the provided frames."""
    if not frames:
        raise ValueError('No frames were provided to export.')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    height, width = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    label = (base_name or 'augmented').rstrip('_')
    for idx in range(video_count):
        video_name = output_dir / f'{label}_no{idx:02d}.mp4'
        writer = cv2.VideoWriter(str(video_name), fourcc, fps, (width, height))
        try:
            if idx == 0:
                video_frames = frames
            else:
                # Sample one transform per video so every frame shares the same effect.
                video_frames = augmentor.augment_sequence_consistently(frames)
            for frame in video_frames:
                writer.write(frame)
        finally:
            writer.release()
        print(f'Wrote {video_name.name}')

In [44]:
# Dataset browser for .mov clips
dataset_root = Path('TDTU-Golf-Pose-v1') / 'Public Test'
mov_files = []
if dataset_root.exists():
    mov_files = sorted(dataset_root.rglob('*.mov'))
    print(f'Found {len(mov_files)} .mov files under {dataset_root}')
else:
    print(f'Create the folder {dataset_root} or update dataset_root to your dataset location.')

selected_mov_path = None
def _format_option(path: Path) -> str:
    try:
        relative_parent = path.parent.relative_to(dataset_root)
        return f"{relative_parent} / {path.name}"
    except ValueError:
        return path.as_posix()

if mov_files:
    mov_dropdown = widgets.Dropdown(
        options=[(_format_option(p), str(p)) for p in mov_files],
        value=str(mov_files[0]),
        description='.mov clip',
        layout=widgets.Layout(width='80%')
    )
    selection_label = widgets.HTML()

    def _set_selected(path_str: str):
        global selected_mov_path
        selected_mov_path = Path(path_str)
        try:
            rel_path = selected_mov_path.relative_to(dataset_root)
            selection_label.value = f'Selected: <code>{rel_path}</code>'
        except ValueError:
            selection_label.value = f'Selected: <code>{selected_mov_path}</code>'

    _set_selected(mov_dropdown.value)
    mov_dropdown.observe(lambda change: _set_selected(change['new']) if change['name'] == 'value' else None, names='value')
    display(widgets.VBox([widgets.HTML('<b>Choose a source .mov clip</b>'), mov_dropdown, selection_label]))
else:
    print('No .mov files found. Double-check the dataset path or file extensions.')

Found 50 .mov files under TDTU-Golf-Pose-v1\Public Test


In [45]:
# Video workflow (.mov/.mp4 input → augmented .mp4 outputs)
fallback_video = Path('golfer_clip.mp4')
source_video_path = (
    Path(selected_mov_path)
    if 'selected_mov_path' in globals() and selected_mov_path and Path(selected_mov_path).exists()
    else fallback_video
)
sample_every = 1  # Increase to skip frames
max_frames = None  # Set an int to limit frames processed
output_video_dir = Path('augmented_videos')

if source_video_path.exists():
    frames, fps = extract_frames_from_video(
        source_video_path,
        sample_every=sample_every,
        max_frames=max_frames,
    )
    print(f'Loaded {len(frames)} frames at ~{fps:.1f} FPS from {source_video_path.name}')
    target_videos = max(1, int(video_multiplier_slider.value))
    export_augmented_videos(
        frames,
        augmentor,
        video_count=target_videos,
        output_dir=output_video_dir,
        fps=int(fps) if fps else 30,
        base_name=source_video_path.stem,
    )
    print(
        f'Created {target_videos} augmented video(s) in {output_video_dir.resolve()} '
        f'from the original {source_video_path.suffix} clip.'
    )
else:
    print(
        'Select a clip in the dataset browser above or place a fallback video at '
        f"{fallback_video.resolve()} before running this cell."
    )

Loaded 223 frames at ~30.0 FPS from Backside-8897-2.mov
Wrote Backside-8897-2_no00.mp4
Wrote Backside-8897-2_no01.mp4
Wrote Backside-8897-2_no02.mp4
Wrote Backside-8897-2_no03.mp4
Created 4 augmented video(s) in D:\CODE\datathon\augmented_videos from the original .mov clip.


In [46]:
# Bulk augmentation over the entire dataset hierarchy
bulk_output_root = Path('augmented_dataset')
bulk_output_root.mkdir(parents=True, exist_ok=True)
bulk_sample_every = 1
bulk_max_frames = None  # Set to an int if you want to cap frames per clip
bulk_target_videos = max(1, int(video_multiplier_slider.value))

if not dataset_root.exists():
    print(f'Dataset root {dataset_root} does not exist. Run the dataset browser cell to configure the path.')
elif not mov_files:
    print('No source .mov clips found. Populate the dataset before running bulk augmentation.')
else:
    processed = 0
    failures: list[tuple[Path, str]] = []
    for clip_path in mov_files:
        try:
            rel_parent = clip_path.parent.relative_to(dataset_root)
            clip_output_dir = bulk_output_root / rel_parent / clip_path.stem
            clip_output_dir.mkdir(parents=True, exist_ok=True)
            frames, fps = extract_frames_from_video(
                clip_path,
                sample_every=bulk_sample_every,
                max_frames=bulk_max_frames,
            )
            export_augmented_videos(
                frames,
                augmentor,
                video_count=bulk_target_videos,
                output_dir=clip_output_dir,
                fps=int(fps) if fps else 30,
                base_name=clip_path.stem,
            )
            processed += 1
        except Exception as exc:
            failures.append((clip_path, str(exc)))
            print(f'[WARN] Failed on {clip_path.name}: {exc}')

    print(f'Bulk augmentation finished. Successful clips: {processed}/{len(mov_files)}')
    if failures:
        print('Failures:')
        for clip_path, message in failures:
            print(f'  - {clip_path}: {message}')

Wrote Backside-8897-2_no00.mp4
Wrote Backside-8897-2_no01.mp4
Wrote Backside-8897-2_no02.mp4
Wrote Backside-8897-2_no03.mp4
Wrote Backside-8900-11_no00.mp4
Wrote Backside-8900-11_no01.mp4
Wrote Backside-8900-11_no02.mp4
Wrote Backside-8900-11_no03.mp4
Wrote Side-6088-2_no00.mp4
Wrote Side-6088-2_no01.mp4
Wrote Side-6088-2_no02.mp4
Wrote Side-6088-2_no03.mp4
Wrote Side-6089-1_no00.mp4
Wrote Side-6089-1_no01.mp4
Wrote Side-6089-1_no02.mp4
Wrote Side-6089-1_no03.mp4
Wrote Backside-8900-8_no00.mp4
Wrote Backside-8900-8_no01.mp4
Wrote Backside-8900-8_no02.mp4
Wrote Backside-8900-8_no03.mp4
Wrote Backside-8900-9_no00.mp4
Wrote Backside-8900-9_no01.mp4
Wrote Backside-8900-9_no02.mp4
Wrote Backside-8900-9_no03.mp4
Wrote Side-6086-8_no00.mp4
Wrote Side-6086-8_no01.mp4
Wrote Side-6086-8_no02.mp4
Wrote Side-6086-8_no03.mp4
Wrote Side-6086-9_no00.mp4
Wrote Side-6086-9_no01.mp4
Wrote Side-6086-9_no02.mp4
Wrote Side-6086-9_no03.mp4
Wrote Backside-8896-22_no00.mp4
Wrote Backside-8896-22_no01.mp4
Wrot